# 04 — Churn Classification


**Decision Log alignment:**
- `CHURN_WINDOW_DAYS = 90` — client with no purchase in the 90-day window before the reference date is labelled `churn=1`
- Approach: supervised classification (class imbalance handled via `scale_pos_weight` / `class_weight='balanced'`)
- Primary metric: **PR-AUC** (more informative than ROC-AUC for imbalanced classes)
- Champion expected: XGBoost

In [ ]:
import sys, warnings, os
warnings.filterwarnings('ignore')

from pathlib import Path
COFFEE_ROOT = Path().resolve().parent
sys.path.insert(0, str(COFFEE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import mlflow
import mlflow.sklearn
import mlflow.xgboost

MLFLOW_TRACKING_URI = str(COFFEE_ROOT / 'mlruns')
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
mlflow.set_experiment("churn_classification")
print('MLflow tracking URI:', mlflow.get_tracking_uri())

## Section 1 — Load Data & Class Distribution

In [ ]:
import subprocess, sys as _sys
# Build churn_features.parquet if it does not exist yet
features_path = COFFEE_ROOT / 'data' / 'features' / 'churn_features.parquet'
if not features_path.exists():
    print('Building churn_features.parquet …')
    subprocess.run([_sys.executable, '-m', 'src.features.churn_features'],
                   cwd=str(COFFEE_ROOT), check=True)

df = pd.read_parquet(features_path)
print(f'Dataset: {len(df):,} customers  |  columns: {df.columns.tolist()}')
df.head()

In [ ]:
# Class distribution
counts = df['churn'].value_counts().sort_index()
labels = {0: 'Active (0)', 1: 'Churned (1)'}
print('Class distribution:')
for k, v in counts.items():
    print(f'  {labels[k]}: {v:,}  ({v/len(df):.1%})')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar([labels[0], labels[1]], counts.values, color=['#3b82d4', '#e05252'])
axes[0].set_title('Churn Class Distribution')
axes[0].set_ylabel('Customers')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=11)

axes[1].pie(counts.values, labels=[labels[k] for k in counts.index],
            autopct='%1.1f%%', colors=['#3b82d4', '#e05252'], startangle=90)
axes[1].set_title('Proportion')

plt.tight_layout()
plt.savefig(str(COFFEE_ROOT / 'data' / 'predictions' / 'churn_class_distribution.png'), dpi=100)
plt.show()
print('Class balance ratio (neg/pos):', counts[0] / counts[1])

In [ ]:
# Descriptive statistics by class
numeric_cols = ['recency_days', 'frequency', 'monetary',
                'recency_norm', 'frequency_norm', 'monetary_norm',
                'freq_last_4w', 'freq_last_8w', 'avg_ticket_recent',
                'n_distinct_products', 'days_since_first_purchase']
desc_by_class = df.groupby('churn')[numeric_cols].mean().T
desc_by_class.columns = ['Active (churn=0)', 'Churned (churn=1)']
desc_by_class['diff_%'] = ((desc_by_class['Active (churn=0)'] - desc_by_class['Churned (churn=1)'])
                            / desc_by_class['Active (churn=0)'].replace(0, np.nan) * 100).round(1)
print('Mean feature values by class:')
print(desc_by_class.round(3).to_string())

## Section 2 — Preparation: Encoding, Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# One-hot encode segment_name
df_model = pd.get_dummies(df, columns=['segment_name'], drop_first=False)

FEATURE_COLS = (
    ['recency_norm', 'frequency_norm', 'monetary_norm',
     'recency_days', 'frequency', 'monetary',
     'cluster_id',
     'freq_last_4w', 'freq_last_8w', 'avg_ticket_recent',
     'n_distinct_products', 'days_since_first_purchase']
    + [c for c in df_model.columns if c.startswith('segment_name_')]
)

X = df_model[FEATURE_COLS]
y = df_model['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,} samples  |  Test: {len(X_test):,} samples')
print(f'Train churn rate: {y_train.mean():.1%}  |  Test churn rate: {y_test.mean():.1%}')
print(f'Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}')

## Section 3 — Baseline: Logistic Regression

In [ ]:
from src.models.churn.baseline import train_logistic
from src.models.churn.evaluate import evaluate

with mlflow.start_run(run_name='logistic_regression_baseline') as lr_run:
    lr_model = train_logistic(X_train, y_train)
    lr_metrics = evaluate(lr_model, X_test, y_test)

    mlflow.log_params({'model': 'LogisticRegression', 'class_weight': 'balanced',
                       'threshold': 0.5})
    mlflow.log_metrics(lr_metrics)
    mlflow.sklearn.log_model(lr_model, 'model')
    lr_run_id = lr_run.info.run_id

print('Logistic Regression metrics:')
for k, v in lr_metrics.items():
    print(f'  {k}: {v:.4f}')

## Section 4 — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name='random_forest') as rf_run:
    rf_model = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    )
    rf_model.fit(X_train, y_train)
    rf_metrics = evaluate(rf_model, X_test, y_test)

    mlflow.log_params({'model': 'RandomForest', 'n_estimators': 300,
                       'max_depth': 8, 'class_weight': 'balanced', 'threshold': 0.5})
    mlflow.log_metrics(rf_metrics)
    mlflow.sklearn.log_model(
        rf_model,
        'model',
        skops_trusted_types=['sklearn.tree._tree.Tree']
    )
    rf_run_id = rf_run.info.run_id

print('Random Forest metrics:')
for k, v in rf_metrics.items():
    print(f'  {k}: {v:.4f}')

## Section 5 — XGBoost

In [ ]:
from src.models.churn.train import train as train_xgb

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight = {scale_pos_weight:.2f}')

with mlflow.start_run(run_name='xgboost_churn') as xgb_run:
    xgb_model = train_xgb(X_train, y_train)
    xgb_metrics = evaluate(xgb_model, X_test, y_test)

    mlflow.log_params({'model': 'XGBoost', 'scale_pos_weight': round(scale_pos_weight, 2),
                       'n_estimators': 300, 'threshold': 0.5})
    mlflow.log_metrics(xgb_metrics)
    mlflow.xgboost.log_model(xgb_model, 'model')
    xgb_run_id = xgb_run.info.run_id

print('XGBoost metrics:')
for k, v in xgb_metrics.items():
    print(f'  {k}: {v:.4f}')

## Section 6 — Champion Selection (by PR-AUC)

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Precision': [lr_metrics['precision'], rf_metrics['precision'], xgb_metrics['precision']],
    'Recall':    [lr_metrics['recall'],    rf_metrics['recall'],    xgb_metrics['recall']],
    'F1':        [lr_metrics['f1'],        rf_metrics['f1'],        xgb_metrics['f1']],
    'ROC-AUC':   [lr_metrics['roc_auc'],   rf_metrics['roc_auc'],   xgb_metrics['roc_auc']],
    'PR-AUC':    [lr_metrics['pr_auc'],    rf_metrics['pr_auc'],    xgb_metrics['pr_auc']],
})
comparison = comparison.sort_values('PR-AUC', ascending=False).reset_index(drop=True)
print('=== Model Comparison (sorted by PR-AUC) ===')
print(comparison.round(4).to_string(index=False))

champion_name = comparison.iloc[0]['Model']
print(f'\nChampion: {champion_name}  (PR-AUC = {comparison.iloc[0]["PR-AUC"]:.4f})')

In [ ]:
# Map champion name to model object / run_id
_champion_map = {
    'Logistic Regression': (lr_model, lr_run_id, 'sklearn'),
    'Random Forest': (rf_model, rf_run_id, 'sklearn'),
    'XGBoost': (xgb_model, xgb_run_id, 'xgboost'),
}
champion_model, champion_run_id, champion_flavor = _champion_map[champion_name]
TUNED_RUN_NAME = champion_name.lower().replace(' ', '_') + '_tuned'
print(f'Champion run ID: {champion_run_id}')
print(f'Tuned run name will be: {TUNED_RUN_NAME}')

## Section 7 — Hyperparameter Tuning via Optuna (≥20 trials)

In [ ]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)

NEG = int((y_train == 0).sum())
POS = int((y_train == 1).sum())
SPW = NEG / max(POS, 1)

def _make_model(trial, name: str):
    if name == 'XGBoost':
        return XGBClassifier(
            n_estimators=trial.suggest_int('n_estimators', 100, 600),
            max_depth=trial.suggest_int('max_depth', 3, 8),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            subsample=trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
            min_child_weight=trial.suggest_int('min_child_weight', 1, 10),
            gamma=trial.suggest_float('gamma', 0, 5),
            scale_pos_weight=SPW,
            eval_metric='aucpr',
            random_state=42,
            verbosity=0,
            n_jobs=-1,
        )
    elif name == 'Random Forest':
        return RandomForestClassifier(
            n_estimators=trial.suggest_int('n_estimators', 100, 600),
            max_depth=trial.suggest_int('max_depth', 3, 15),
            min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
            class_weight='balanced',
            random_state=42,
            n_jobs=-1,
        )
    else:  # Logistic Regression
        return Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(
                C=trial.suggest_float('C', 0.001, 100, log=True),
                class_weight='balanced',
                max_iter=2000,
                random_state=42,
            )),
        ])

def objective(trial):
    model = _make_model(trial, champion_name)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train,
                             cv=cv, scoring='average_precision', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20, show_progress_bar=False)

print(f'Best PR-AUC (CV): {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

In [ ]:
# Retrain champion on full training set with best params
best_params = study.best_params.copy()

if champion_name == 'XGBoost':
    best_params['scale_pos_weight'] = SPW
    best_params['eval_metric'] = 'aucpr'
    best_params['random_state'] = 42
    best_params['verbosity'] = 0
    best_params['n_jobs'] = -1
    tuned_model = XGBClassifier(**best_params)
elif champion_name == 'Random Forest':
    best_params['class_weight'] = 'balanced'
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1
    tuned_model = RandomForestClassifier(**best_params)
else:
    tuned_model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=best_params['C'],
                                   class_weight='balanced',
                                   max_iter=2000, random_state=42)),
    ])

tuned_model.fit(X_train, y_train)
tuned_metrics = evaluate(tuned_model, X_test, y_test)

with mlflow.start_run(run_name=TUNED_RUN_NAME) as tuned_run:
    mlflow.log_params({**best_params, 'model': champion_name, 'n_optuna_trials': 20})
    mlflow.log_metrics(tuned_metrics)
    if champion_flavor == 'xgboost':
        mlflow.xgboost.log_model(tuned_model, 'model')
    else:
        mlflow.sklearn.log_model(
            tuned_model,
            'model',
            skops_trusted_types=['sklearn.tree._tree.Tree']
        )
    tuned_run_id = tuned_run.info.run_id

print(f'Tuned {champion_name} metrics:')
for k, v in tuned_metrics.items():
    print(f'  {k}: {v:.4f}')
print(f'Improvement over base: PR-AUC {tuned_metrics["pr_auc"] - comparison.iloc[0]["PR-AUC"]:+.4f}')

## Section 8 — Feature Importance + SHAP

In [ ]:
import shap

# Feature importance from the champion model
if hasattr(tuned_model, 'feature_importances_'):
    importances = tuned_model.feature_importances_
elif hasattr(tuned_model, 'named_steps'):
    # Pipeline — get from last step if possible
    last_step = list(tuned_model.named_steps.values())[-1]
    if hasattr(last_step, 'coef_'):
        importances = np.abs(last_step.coef_[0])
    else:
        importances = np.ones(len(FEATURE_COLS))
else:
    importances = np.ones(len(FEATURE_COLS))

feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
top_n = feat_imp.head(12)
ax.barh(top_n.index[::-1], top_n.values[::-1], color='#3b82d4')
ax.set_title(f'Feature Importance — {champion_name} (Tuned)', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(str(COFFEE_ROOT / 'data' / 'predictions' / 'feature_importance.png'), dpi=100)
plt.show()

print('\nTop 5 most important features:')
for i, (feat, imp) in enumerate(feat_imp.head(5).items(), 1):
    print(f'  {i}. {feat}: {imp:.4f}')

In [ ]:
# SHAP beeswarm summary
X_test_arr = np.array(X_test, dtype=float)

try:
    if champion_name == 'XGBoost':
        explainer = shap.TreeExplainer(tuned_model)
        shap_values = explainer.shap_values(X_test_arr)
    elif champion_name == 'Random Forest':
        explainer = shap.TreeExplainer(tuned_model)
        shap_values = explainer.shap_values(X_test_arr)
        # For binary RF, shap_values is a list [class0, class1]
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
    else:
        # Logistic Regression pipeline
        X_scaled = tuned_model.named_steps['scaler'].transform(X_test_arr)
        explainer = shap.LinearExplainer(
            tuned_model.named_steps['clf'],
            X_scaled,
            feature_perturbation='interventional',
        )
        shap_values = explainer.shap_values(X_scaled)

    fig_shap, ax_shap = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS,
                      plot_type='dot', show=False)
    plt.title(f'SHAP Summary (beeswarm) — {champion_name} Tuned', fontsize=12)
    plt.tight_layout()
    plt.savefig(str(COFFEE_ROOT / 'data' / 'predictions' / 'shap_summary.png'),
                dpi=100, bbox_inches='tight')
    plt.show()
    print('SHAP plot saved.')
except Exception as e:
    print(f'SHAP computation skipped: {e}')

## Section 9 — Threshold Analysis

In [ ]:
from sklearn.metrics import precision_recall_curve, auc

y_prob_test = tuned_model.predict_proba(X_test)[:, 1]
prec_curve, rec_curve, thresh_curve = precision_recall_curve(y_test, y_prob_test)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rec_curve, prec_curve, lw=2, color='#3b82d4', label=f'PR curve (AUC={tuned_metrics["pr_auc"]:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(COFFEE_ROOT / 'data' / 'predictions' / 'pr_curve.png'), dpi=100)
plt.show()

# Threshold table
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_rows = []
for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_prob_test >= thr).astype(int)
    threshold_rows.append({
        'Threshold': thr,
        'Precision': precision_score(y_test, y_pred_t, zero_division=0),
        'Recall':    recall_score(y_test, y_pred_t, zero_division=0),
        'F1':        f1_score(y_test, y_pred_t, zero_division=0),
        'N_flagged': int(y_pred_t.sum()),
    })

thresh_df = pd.DataFrame(threshold_rows)
print(thresh_df.round(4).to_string(index=False))

In [ ]:
# Business recommendation: maximise recall for high-value retention campaigns
# while keeping precision > 0.5 to avoid wasting campaign budget
# Threshold 0.4 balances recall (catches more churners) with reasonable precision
RECOMMENDED_THRESHOLD = 0.4

rec_row = thresh_df[thresh_df['Threshold'] == RECOMMENDED_THRESHOLD].iloc[0]
print(f'Recommended threshold: {RECOMMENDED_THRESHOLD}')
print(f'  Precision: {rec_row["Precision"]:.3f}')
print(f'  Recall:    {rec_row["Recall"]:.3f}')
print(f'  F1:        {rec_row["F1"]:.3f}')
print(f'  Customers flagged: {int(rec_row["N_flagged"])}')
print('\nRationale: higher recall ensures fewer high-value churners are missed.')
print('False positive cost (unnecessary campaign) < False negative cost (lost customer CLV).')

## Section 10 — Generate Predictions for All Customers

In [ ]:
(COFFEE_ROOT / 'data' / 'predictions').mkdir(parents=True, exist_ok=True)

# Apply model to ALL customers (full dataset, not just test set)
df_all = pd.read_parquet(features_path)
df_all_enc = pd.get_dummies(df_all, columns=['segment_name'], drop_first=False)

# Align columns with training set (handle any missing dummies)
for col in FEATURE_COLS:
    if col not in df_all_enc.columns:
        df_all_enc[col] = 0

X_all = df_all_enc[FEATURE_COLS]
churn_proba = tuned_model.predict_proba(X_all)[:, 1]
churn_label = (churn_proba >= RECOMMENDED_THRESHOLD).astype(int)

# Bring back segment_name and monetary for business prioritisation
predictions = pd.DataFrame({
    'card': df_all['card'].values,
    'churn_probability': churn_proba,
    'churn_label': churn_label,
    'segment_name': df_all['segment_name'].values,
    'monetary': df_all['monetary'].values,
})
predictions = predictions.sort_values('churn_probability', ascending=False).reset_index(drop=True)

out_path = COFFEE_ROOT / 'data' / 'predictions' / 'churn_predictions.parquet'
predictions.to_parquet(out_path, index=False)

print(f'Saved {len(predictions):,} predictions to {out_path}')
print(f'Flagged as churn (threshold={RECOMMENDED_THRESHOLD}): {churn_label.sum():,} customers')
print(predictions.head(10).to_string(index=False))

In [ ]:
# High-value churners: top 10 by monetary value
high_value_churners = predictions[predictions['churn_label'] == 1].nlargest(10, 'monetary')
print('Top 10 high-value customers at churn risk:')
print(high_value_churners[['card', 'churn_probability', 'segment_name', 'monetary']].to_string(index=False))

## Section 11 — Register Champion in MLflow Model Registry

In [ ]:
from mlflow.tracking import MlflowClient

MODEL_NAME = 'churn_classifier'
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

# Register model from the tuned run
model_uri = f'runs:/{tuned_run_id}/model'
registered = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
version = registered.version
print(f'Registered model: {MODEL_NAME} v{version}')

# Transition to Production using the new API (aliases)
try:
    # MLflow >= 2.x: use aliases instead of stages
    client.set_registered_model_alias(MODEL_NAME, 'Production', version)
    print(f'Set alias Production → v{version}')
except Exception:
    try:
        client.transition_model_version_stage(
            name=MODEL_NAME, version=version, stage='Production',
            archive_existing_versions=True,
        )
        print(f'Transitioned v{version} to Production')
    except Exception as e2:
        print(f'Stage transition skipped ({e2})')

# Log final summary metrics to the tuned run
with mlflow.start_run(run_id=tuned_run_id):
    mlflow.set_tag('champion', 'true')
    mlflow.set_tag('recommended_threshold', str(RECOMMENDED_THRESHOLD))
    mlflow.set_tag('registry_model_name', MODEL_NAME)
    mlflow.set_tag('registry_version', str(version))

print(f'Champion model: {champion_name} (tuned)')
print(f'PR-AUC: {tuned_metrics["pr_auc"]:.4f}  |  ROC-AUC: {tuned_metrics["roc_auc"]:.4f}')
print(f'Recommended threshold: {RECOMMENDED_THRESHOLD}')
print(f'Predictions saved: data/predictions/churn_predictions.parquet')
print(f'MLflow model registered: {MODEL_NAME} v{version} → Production')